<a href="https://colab.research.google.com/github/fatasa/Tesis/blob/main/augmentation80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')

#PATH GOOGLE DRIVE
DATASET_DRIVE_PATH = '/content/drive/MyDrive/cell_images'
AUGMENTED_DRIVE_PATH = '/content/drive/MyDrive/cell_images_augmented'

Mounted at /content/drive


In [ ]:
def apply_augmentation(image):
    augmented = []
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)

    # 1. Rotasi 30 derajat
    M = cv2.getRotationMatrix2D(center, 30, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    augmented.append(('rot30', rotated))

    # 2. Horizontal Flip
    h_flip = cv2.flip(image, 1)
    augmented.append(('fliph', h_flip))

    # 3. Vertical Flip
    v_flip = cv2.flip(image, 0)
    augmented.append(('flipv', v_flip))

    return augmented

In [ ]:
def load_raw_data(dataset_dir):
    images = []
    labels = []
    file_names = []
    categories = ['Uninfected', 'Parasitized'] # 0: Uninfected, 1: Parasitized[cite: 1, 2]

    for category in categories:
        folder_path = os.path.join(dataset_dir, category)
        label = 0 if category == 'Uninfected' else 1

        if not os.path.exists(folder_path):
            print(f"⚠️ Warning: Folder {folder_path} tidak ditemukan!")
            continue

        print(f"Memuat citra dari folder: {category}...")
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                img = cv2.imread(img_path)
                if img is not None:
                    images.append(img)
                    labels.append(label)
                    file_names.append(img_name)

    print(f"\nTotal citra mentah berhasil dimuat: {len(images):,} gambar.")
    return images, np.array(labels), file_names

In [ ]:
if __name__ == "__main__":
    X_raw, y_raw, filenames = load_raw_data(DATASET_DRIVE_PATH)

    #Split 80:20
    X_tr, X_te, y_tr, y_te, fn_tr, fn_te = train_test_split(
        X_raw, y_raw, filenames, test_size=0.2, random_state=42, stratify=y_raw
    )

    for split in ["train", "test"]:
        for cat in ["Uninfected", "Parasitized"]:
            os.makedirs(os.path.join(AUGMENTED_DRIVE_PATH, split, cat), exist_ok=True)

    print("\n---> Menyimpan Data Latih Ter-augmentasi ke Google Drive...")
    for img, label, fname in tqdm(zip(X_tr, y_tr, fn_tr), total=len(X_tr), desc="Proses Data Train"):
        cat_dir = os.path.join(AUGMENTED_DRIVE_PATH, "train", "Parasitized" if label == 1 else "Uninfected")

        clean_fname = os.path.basename(fname)
        base_name = os.path.splitext(clean_fname)[0]

        # Validasi format data piksel (uint8)
        img_save = img.astype(np.uint8) if img.dtype != np.uint8 else img

        # Simpan citra asli
        cv2.imwrite(os.path.join(cat_dir, f"{base_name}_ori.png"), img_save)

        # Simpan citra hasil augmentasi (Rotasi, H-Flip, V-Flip)
        for aug_type, aug_img in apply_augmentation(img):
            cv2.imwrite(os.path.join(cat_dir, f"{base_name}_{aug_type}.png"), aug_img)

    print("\n---> Menyimpan Data Uji Murni ke Google Drive...")
    for img, label, fname in tqdm(zip(X_te, y_te, fn_te), total=len(X_te), desc="Proses Data Test"):
        cat_dir = os.path.join(AUGMENTED_DRIVE_PATH, "test", "Parasitized" if label == 1 else "Uninfected")

        clean_fname = os.path.basename(fname)
        img_save = img.astype(np.uint8) if img.dtype != np.uint8 else img

        cv2.imwrite(os.path.join(cat_dir, clean_fname), img_save)

    print(f"\n[SELESAI] Seluruh dataset hasil split & augmentasi tersimpan di:\n{AUGMENTED_DRIVE_PATH}")

Memuat citra dari folder: Uninfected...
Memuat citra dari folder: Parasitized...

Total citra mentah berhasil dimuat: 27,558 gambar.

---> Menyimpan Data Latih Ter-augmentasi ke Google Drive...


Proses Data Train: 100%|██████████| 22046/22046 [26:00<00:00, 14.13it/s]



---> Menyimpan Data Uji Murni ke Google Drive...


Proses Data Test: 100%|██████████| 5512/5512 [02:33<00:00, 35.92it/s]


[SELESAI] Seluruh dataset hasil split & augmentasi tersimpan di:
/content/drive/MyDrive/cell_images_augmented_80


In [ ]:
def count_files_in_dir(folder_path):
    if not os.path.exists(folder_path):
        return 0
    return len([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

In [ ]:
    train_uninfected_cnt = count_files_in_dir(os.path.join(AUGMENTED_DRIVE_PATH, "train", "Uninfected"))
    train_parasitized_cnt = count_files_in_dir(os.path.join(AUGMENTED_DRIVE_PATH, "train", "Parasitized"))
    test_uninfected_cnt = count_files_in_dir(os.path.join(AUGMENTED_DRIVE_PATH, "test", "Uninfected"))
    test_parasitized_cnt = count_files_in_dir(os.path.join(AUGMENTED_DRIVE_PATH, "test", "Parasitized"))

    total_train = train_uninfected_cnt + train_parasitized_cnt
    total_test = test_uninfected_cnt + test_parasitized_cnt
    total_all = total_train + total_test

    print("\n" + "="*55)
    print("      REKAPITULASI JUMLAH DATASET HASIL AUGMENTASI")
    print("="*55)
    print(f"1. Data Latih (Train) [Telah Di-augmentasi]:")
    print(f"   - Uninfected  : {train_uninfected_cnt:,} citra")
    print(f"   - Parasitized : {train_parasitized_cnt:,} citra")
    print(f"   - Subtotal    : {total_train:,} citra")
    print("-" * 55)
    print(f"2. Data Uji (Test) [Murni Tanpa Augmentasi]:")
    print(f"   - Uninfected  : {test_uninfected_cnt:,} citra")
    print(f"   - Parasitized : {test_parasitized_cnt:,} citra")
    print(f"   - Subtotal    : {total_test:,} citra")
    print("-" * 55)
    print(f"TOTAL KESELURUHAN BERKAS CITRA: {total_all:,} citra")
    print("="*55)


      REKAPITULASI JUMLAH DATASET HASIL AUGMENTASI
1. Data Latih (Train) [Telah Di-augmentasi]:
   - Uninfected  : 44,092 citra
   - Parasitized : 44,092 citra
   - Subtotal    : 88,184 citra
-------------------------------------------------------
2. Data Uji (Test) [Murni Tanpa Augmentasi]:
   - Uninfected  : 2,756 citra
   - Parasitized : 2,756 citra
   - Subtotal    : 5,512 citra
-------------------------------------------------------
TOTAL KESELURUHAN BERKAS CITRA: 93,696 citra
